In [ ]:
# Scenario 1: AI Customer Support Multi-Agent System
# 🎯 Objective:
# Build a multi-agent customer support system that can classify, delegate, and resolve user queries.

# 💼 Problem Statement
# A company wants to automate its support system using AI agents.

# The system should:

# Understand customer queries
# Route them to the correct department
# Provide responses
# 👥 Agents to Build
# Classifier Agent
# Identifies query type (Billing / Technical / General)
# Billing Agent
# Handles payment/refund queries
# Technical Support Agent
# Handles app/software issues
# Response Agent
# Combines outputs into a final response

!pip install crewai litellm langchain-community langchain-litellm


import os
from crewai import Agent, Task, Crew
from langchain_litellm import ChatLiteLLM
from google.colab import userdata

# 3. SET API KEY (Groq)

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")


# 4. LLM

llm = "groq/llama-3.3-70b-versatile"

# 5. AGENTS

classifier = Agent(
    role="Query Classifier",
    goal="Classify queries",
    backstory="Expert in intent detection",
    llm="groq/llama-3.3-70b-versatile",
    verbose=False
)

billing_agent = Agent(
    role="Billing Support Agent",
    goal="Resolve billing, refund, and subscription issues politely",
    backstory="Handles payments and subscriptions professionally",
    llm=llm,
    verbose=False
)

technical_agent = Agent(
    role="Technical Support Agent",
    goal="Fix bugs, crashes, and app issues with clear steps",
    backstory="Expert software engineer",
    llm=llm,
    verbose=False
)

response_agent = Agent(
    role="Response Formatter",
    goal="Generate a clean, structured final response",
    backstory="Customer support communication expert",
    llm=llm,
    verbose=False
)

# 6. SMART ROUTING (HYBRID)

def route_query(query):
    q = query.lower()

    # Rule-based (fast)
    if any(word in q for word in ["charge", "refund", "payment", "subscription"]):
        return "billing"
    elif any(word in q for word in ["error", "crash", "bug", "issue"]):
        return "technical"

    # LLM fallback (better accuracy)
    prompt = f"""
    Classify this query into one word:
    Billing / Technical / General

    Query: {query}
    """

    try:
        result = llm.invoke(prompt).content.lower()
        if "billing" in result:
            return "billing"
        elif "technical" in result:
            return "technical"
        else:
            return "general"
    except:
        return "general"

# 7. MAIN SYSTEM

def run_system(user_query):

    category = route_query(user_query)

    # Task 1: Classification
    classify_task = Task(
        description=f"Classify the query: {user_query}",
        expected_output="Billing / Technical / General",
        agent=classifier
    )

    # Task 2: Handling
    if category == "billing":
        handle_task = Task(
            description=f"""
            Resolve this billing issue politely:
            {user_query}
            Provide steps and reassurance.
            """,
            expected_output="Billing solution",
            agent=billing_agent
        )

    elif category == "technical":
        handle_task = Task(
            description=f"""
            Solve this technical issue step-by-step:
            {user_query}
            """,
            expected_output="Technical solution",
            agent=technical_agent
        )

    else:
        handle_task = Task(
            description=f"""
            Answer this general query clearly:
            {user_query}
            """,
            expected_output="Helpful response",
            agent=response_agent
        )

    # Task 3: Final Response Formatting
    final_task = Task(
        description=f"""
        Convert the response into a clean customer support reply.
        Add greeting and closing line.
        """,
        expected_output="Professional final response",
        agent=response_agent,
        context=[handle_task]
    )

    crew = Crew(
        agents=[classifier, billing_agent, technical_agent, response_agent],
        tasks=[classify_task, handle_task, final_task],
        verbose=False
    )

    result = crew.kickoff()

    return category, result

# 8. USER LOOP

print("🤖 AI Customer Support System")
print("Type 'exit' to quit\n")

while True:
    user_query = input("👤 You: ")

    if user_query.lower() == "exit":
        print("👋 Goodbye!")
        break

    try:
        category, response = run_system(user_query)

        print("\n📂 Category:", category.upper())
        print("🤖 Response:\n", response)
        print("\n" + "="*50 + "\n")

    except Exception as e:
        print("⚠️ Error:", str(e))


🤖 AI Customer Support System
Type 'exit' to quit

👤 You: My app crashes

📂 Category: TECHNICAL
🤖 Response:
 Dear Customer,

We appreciate you reaching out to us regarding the app crash issue you're experiencing. To assist you in resolving this problem, we've put together a step-by-step troubleshooting guide. Please follow these steps carefully to identify and potentially fix the issue.

### Step 1: Gather Information
To better understand the issue, please provide more details about the problem, such as:
- The name of the app that's crashing
- The device and operating system you're using (e.g., Android, iOS, Windows)
- The version of the app and the operating system
- Any error messages or codes that appear when the app crashes
- The actions you took leading up to the crash

### Step 2: Basic Troubleshooting
1. **Restart Your Device**: Sometimes, simply restarting your device can resolve the issue.
2. **Check for Updates**: Ensure your app and operating system are up to date, as updates

╭─────────────────────────────────────────────── Execution Traces ────────────────────────────────────────────────╮
│                                                                                                                 │
│  🔍 Detailed execution traces are available!                                                                    │
│                                                                                                                 │
│  View insights including:                                                                                       │
│    • Agent decision-making process                                                                              │
│    • Task execution flow and timing                                                                             │
│    • Tool usage details                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Would you like to view your execution traces? [y/N] (20s timeout): 

In [ ]:
Architecture diagram:


          User Query
               ↓
         Classifier Agent
               ↓
   ┌───────────┼────────────┐
   ↓           ↓            ↓
 Billing    Technical    General
 Agent        Agent        Agent
   ↓           ↓            ↓
        🧾 Response Agent
               ↓
         Final Answer
